# Concatenating multiple feature extraction methods: scikit-learn vs Moxie

**Reference example:** `../scikit-learn/examples/compose/plot_feature_union.py` at scikit-learn
`4358542a1bd39af2e7e1cbace8b9fed067473c5b`
([rendered gallery page](https://scikit-learn.org/dev/auto_examples/compose/plot_feature_union.html)).

**Scenario:** on the bundled iris data, a `FeatureUnion` of `PCA(n_components=2)` and `SelectKBest(k=1)` builds a
3-column feature space; a `Pipeline(features, SVC(kernel="linear"))` is then tuned with `GridSearchCV` over
`features__pca__n_components`, `features__univ_select__k` and `svm__C`.

**Adaptations:**
- Moxie builds the union with `make_union`, so branch names are `pca` and `selectkbest` instead of the
  example's explicit `univ_select`; the Python reference below uses the original names for the run and
  `make_union` once for the names comparison.
- **Pipeline(union, SVC) is a GAP:** Moxie's `FeatureUnion` target is `NDArray[float64]` (what `SelectKBest`
  needs), while `SVC` takes typed `List[Label]` targets. Pipeline therefore fits the union with `y=None` and
  `SelectKBest` fails clearly (proved by the `--expect-fail` cell). The Moxie run fits the union and SVC as two
  explicit stages, which is what `Pipeline.fit` does in sklearn.
- **GridSearchCV is a GAP:** Moxie's search needs a hand-written typed `SearchAdapter` per estimator, and
  none exists for this pipeline. Instead, both sides walk the same 18-point grid through **nested typed
  `set_params`** on the union (`params.transformer_list.first.estimator.n_components`, ...), refit and report
  training accuracy. That is a deterministic stand-in for the search, not cross-validation.
- `verbose=10` search logging is omitted.

**Determinism:** deterministic (iris, full-SVD PCA, libsvm-style SMO with no random state for `kernel="linear"`).

In [1]:
import sys
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "notebooks/_support/moxie_nb.py").exists())
sys.path.insert(0, str(ROOT / "notebooks/_support"))
%load_ext moxie_nb
nb = moxie_nb.Session("pipeline/plot_feature_union")

## 1. scikit-learn (Python reference)

The example code, unchanged except for printing a compact search result.

In [2]:
import numpy as np
from sklearn.datasets import load_iris
from sklearn.decomposition import PCA
from sklearn.feature_selection import SelectKBest
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import FeatureUnion, Pipeline, make_union
from sklearn.svm import SVC

iris = load_iris()

X, y = iris.data, iris.target

pca = PCA(n_components=2)

selection = SelectKBest(k=1)

combined_features = FeatureUnion([("pca", pca), ("univ_select", selection)])

X_features = combined_features.fit(X, y).transform(X)
print("Combined space has", X_features.shape[1], "features")

svm = SVC(kernel="linear")

pipeline = Pipeline([("features", combined_features), ("svm", svm)])

param_grid = dict(
    features__pca__n_components=[1, 2, 3],
    features__univ_select__k=[1, 2],
    svm__C=[0.1, 1, 10],
)

grid_search = GridSearchCV(pipeline, param_grid=param_grid)
grid_search.fit(X, y)
print(grid_search.best_estimator_)
print("best params:", grid_search.best_params_, "best CV score:", round(grid_search.best_score_, 4))

# Reference values for the comparison below.
sk_names = make_union(PCA(n_components=2), SelectKBest(k=1)).fit(X, y).get_feature_names_out()
pipeline.set_params(features__pca__n_components=2, features__univ_select__k=1, svm__C=1).fit(X, y)
sk_decision = pipeline.decision_function(X)
sk_grid = []
for nc in [1, 2, 3]:
    for k in [1, 2]:
        for c in [0.1, 1, 10]:
            pipeline.set_params(features__pca__n_components=nc, features__univ_select__k=k, svm__C=c)
            sk_grid.append(pipeline.fit(X, y).score(X, y))
sk_grid = np.array(sk_grid)
nb.share(X=X, y=y.astype(np.float64))

Combined space has 3 features


Pipeline(steps=[('features',
                 FeatureUnion(transformer_list=[('pca', PCA(n_components=3)),
                                                ('univ_select',
                                                 SelectKBest(k=1))])),
                ('svm', SVC(C=10, kernel='linear'))])
best params: {'features__pca__n_components': 3, 'features__univ_select__k': 1, 'svm__C': 10} best CV score: 0.98


## 2. Moxie (native Mojo)

Union via `make_union`, feature names, an explicit SVC stage, and the grid walked with nested typed `set_params`.

In [3]:
%%mojo feature_union
from numojo import NDArray, Shape
from moxie.pipeline import Pipeline, FeatureUnion, UnionPair, union_step, make_union
from moxie.decomposition import PCA
from moxie.feature_selection import SelectKBest, SelectionParameter
from moxie.svm import SVC
from moxie_nb_io import load, save, save_floats, save_strings


def labels(y: NDArray[DType.float64]) raises -> List[Float64]:
    var out = List[Float64]()
    for i in range(y.shape[0]):
        out.append(Float64(y.item(i)))
    return out^


def main() raises:
    var X = load("X")
    var y = load("y")
    var y_labels = labels(y)
    var combined = make_union(PCA(n_components=2), SelectKBest(k=SelectionParameter(1)))
    _ = combined.fit(X, y.copy())
    var X_features = combined.transform(X)
    print("Combined space has", X_features.shape[1], "features")
    save("X_features", X_features)
    var svm = SVC(kernel="linear")
    _ = svm.fit(X_features, y_labels)
    save("decision", svm.decision_function(X_features))
    var scores = List[Float64]()
    var params = combined.get_params()
    for nc in [1, 2, 3]:
        for k in [1, 2]:
            for c in [0.1, 1.0, 10.0]:
                params.transformer_list.first.estimator.n_components = nc
                params.transformer_list.second.estimator.param = SelectionParameter(k)
                _ = combined.set_params(params)
                var features = combined.fit_transform(X, y.copy())
                var model = SVC(kernel="linear", C=c)
                _ = model.fit(features, y_labels)
                scores.append(model.score(features, y_labels))
    save_floats("grid_train_scores", scores)


Combined space has 3 features


### Unsupported: `Pipeline(FeatureUnion(PCA, SelectKBest), SVC)`

The union's `NDArray` target cannot carry SVC's `List[Label]` labels, so the pipeline fits the unsupervised
union with `y=None` and the supervised `SelectKBest` branch refuses to fit. The cell must fail.

In [4]:
%%mojo feature_union_pipeline --expect-fail
from numojo import NDArray
from moxie.pipeline import Pipeline, make_union
from moxie.decomposition import PCA
from moxie.feature_selection import SelectKBest, SelectionParameter
from moxie.svm import SVC
from moxie_nb_io import load


def main() raises:
    var X = load("X")
    var y = load("y")
    var labels = List[Float64]()
    for i in range(y.shape[0]):
        labels.append(Float64(y.item(i)))
    var pipe = Pipeline(make_union(PCA(n_components=2), SelectKBest(k=SelectionParameter(1))), SVC(kernel="linear"))
    _ = pipe.fit(X, labels)

Expected failure (unsupported in Moxie):
exit status 1
stack trace was not collected. Enable stack trace collection with environment variable `MODULAR_DEBUG=stack-trace-on-error`
Unhandled exception caught during execution: ValueError: selected scorer requires y



## 3. Side by side and parity

In [5]:
mx = nb.results["feature_union"]
moxie_nb.side_by_side(sklearn=X_features[:5], moxie=mx["X_features"][:5])
print("sklearn make_union feature names:", list(sk_names))
nb.note("FeatureUnion.get_feature_names_out with PCA/SelectKBest", "GAP",
        "PCA defines get_feature_names_out but does not declare the shared FeatureNamesOut trait; SelectKBest "
        "uses a List[String] signature instead of Optional[List[String]]. The union raises an AttributeError-style "
        "error (owners: decomposition, feature_selection). Union naming itself is covered by tests/pipeline parity.")
nb.compare("union transform shape", X_features.shape, mx["X_features"].shape, exact=True)
nb.compare("union transform values", X_features, mx["X_features"], rtol=1e-10, atol=1e-12,
           reason="PCA via SVD of the same centred matrix with sklearn's svd_flip sign rule; "
                  "different LAPACK/Jacobi rounding stays ~1e-14, SelectKBest column copy is exact")
nb.compare("SVC decision_function on union features", sk_decision, mx["decision"], rtol=1e-6, atol=1e-6,
           reason="same SMO working-set algorithm stops at tol=1e-3 on the dual; both sides take the "
                  "same steps, so remaining differences are floating-point rounding")
nb.compare("training accuracy over the 18-point grid", sk_grid, mx["grid_train_scores"], exact=True)
nb.note("Pipeline(FeatureUnion with SelectKBest, SVC)", "GAP",
        "FeatureUnion target is NDArray[dtype]; SVC targets are List[Label]. The union is fitted with y=None "
        "and the supervised branch fails (see --expect-fail cell). Unsupervised branches work.")
nb.note("GridSearchCV over a Pipeline", "GAP",
        "moxie.model_selection searches need a hand-written typed SearchAdapter; no adapter for Pipeline "
        "exists, and string 'step__param' grids are replaced by typed nested Params records.")
nb.report()
nb.assert_parity()

sklearn make_union feature names: ['pca__pca0', 'pca__pca1', 'selectkbest__x2']


check,status,sklearn shape,moxie shape,max |diff|,max rel,tolerance,detail,tolerance reason
FeatureUnion.get_feature_names_out with PCA/SelectKBest,GAP,,,,,,"PCA defines get_feature_names_out but does not declare the shared FeatureNamesOut trait; SelectKBest uses a List[String] signature instead of Optional[List[String]]. The union raises an AttributeError-style error (owners: decomposition, feature_selection). Union naming itself is covered by tests/pipeline parity.",
union transform shape,PASS,"(2,)","(2,)",,,exact,0 mismatches,
union transform values,PASS,"(150, 3)","(150, 3)",1.42e-14,1.17e-12,"rtol=1e-10, atol=1e-12",,"PCA via SVD of the same centred matrix with sklearn's svd_flip sign rule; different LAPACK/Jacobi rounding stays ~1e-14, SelectKBest column copy is exact"
SVC decision_function on union features,PASS,"(150, 3)","(150, 3)",1.95e-14,1.69e-14,"rtol=1e-06, atol=1e-06",,"same SMO working-set algorithm stops at tol=1e-3 on the dual; both sides take the same steps, so remaining differences are floating-point rounding"
training accuracy over the 18-point grid,PASS,"(18,)","(18,)",,,exact,0 mismatches,
"Pipeline(FeatureUnion with SelectKBest, SVC)",GAP,,,,,,FeatureUnion target is NDArray[dtype]; SVC targets are List[Label]. The union is fitted with y=None and the supervised branch fails (see --expect-fail cell). Unsupervised branches work.,
GridSearchCV over a Pipeline,GAP,,,,,,"moxie.model_selection searches need a hand-written typed SearchAdapter; no adapter for Pipeline exists, and string 'step__param' grids are replaced by typed nested Params records.",
